In [1]:
!python -m pip install pandas jupyter


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Install Dependencies

Install the required packages if they are not already available in the environment.

# 01 — Setup Database

This notebook loads the raw Superstore CSV into a SQLite database.  
All downstream notebooks (`02_sql_analysis`, `03_visualization`) will query from this database instead of reading the CSV directly — this ensures a single source of truth and keeps any data changes in one place.

**Steps:**
1. Install dependencies
2. Load raw CSV into a Pandas DataFrame
3. Validate the data (shape, types, missing values)
4. Write the DataFrame into a SQLite database as the `orders` table
5. Verify the table was created correctly

In [1]:
import pandas as pd
import sqlite3
import os

## Step 2 — Load Raw CSV

We use `encoding="latin1"` because the Superstore CSV contains special characters (e.g. accented letters in product names) that are not UTF-8 compatible.

In [2]:
df = pd.read_csv("../data/superstore.csv", encoding="latin1")
print(f"Shape: {df.shape}")
df.head()

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 3 — Validate the Data

Before writing to the database, we check:
- **Data types** — are numeric columns (Sales, Profit, Discount) correctly parsed as `float`?
- **Missing values** — are there any nulls that need to be handled before analysis?

This step confirms the raw data is clean enough to load as-is.

In [3]:
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Missing values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


## Step 4 — Write to SQLite Database

We write the DataFrame to a SQLite file (`superstore.db`) as a table named `orders`.  
Using `if_exists="replace"` means re-running this notebook will overwrite the table cleanly, so the database always reflects the current CSV without manual cleanup.

In [4]:
conn = sqlite3.connect("../data/superstore.db")

df.to_sql("orders", conn, if_exists="replace", index=False)

print("Database created successfully!")
print(f"Table 'orders' has {len(df)} rows")

Database created successfully!
Table 'orders' has 9994 rows


## Step 5 — Verify

Run a quick `COUNT(*)` query to confirm the table was written correctly.  
The expected row count is **9,994** (matching the original CSV).

In [5]:
test_query = "Select count(*) as total_row FROM orders"
result=pd.read_sql(test_query, conn)
print(result)

   total_row
0       9994


In [6]:
conn.close()
print("Connection closed. setup complete!")

Connection closed. setup complete!
